Python offers a few ways to build a simple class that is just a collection of fields, with
little or no extra functionality. That pattern is known as a “data class”—

1. collections.namedtuple (The Classic)
This is the "old school" way. It creates a subclass of a standard tuple.

Pros: Very memory efficient and fast. You can access data by name (e.g., p.x) or by index (e.g., p[0]).

Cons: It’s immutable, meaning once you create it, you can't change the values.

2. typing.NamedTuple (The Modern Tuple)
This is essentially the same as above but uses Python’s modern type hints.

Difference: It looks like a regular class definition. It’s much more readable and plays nicely with code editors that check for errors.

Constraint: Like its predecessor, it is still a tuple, so it is immutable.

3. @dataclasses.dataclass (The Powerhouse)
This is a decorator added in Python 3.7 that turns a regular class into a data container automatically.

Pros: By default, it is mutable (you can change values). It’s highly customizable—you can easily tell it to be "frozen" (immutable) or to automatically handle complex comparisons.

When to use: Use this for most modern Python projects where you need more flexibility than a simple tuple provides.

In [ ]:
#Vectors comparision
from collections import namedtuple
from typing import NamedTuple
from dataclasses import dataclass

# Fast, but no type hints by default.
vector1 = namedtuple('vector1', ['x','y'])
v1 = vector1(10,y=20)
v1

# Looks like a class, behaves like a tuple. Immutable.
class vector2(NamedTuple):
    x: int
    y: int
v2 = vector2(10,20)
v2

# A real class. Values can be changed after creation.
@dataclass
class vector3:
    x: int
    y: int
v3 = vector3(10,20)
v3

vector3(x=10, y=20)

In [37]:
v11 = vector1(10,20)
v1 == v11
#Meaningful __eq__


True

In [27]:
#Unpacking: You can do a, b = v1 or a, b = v2 because they are tuples. You cannot do that with a @dataclass without extra code.
from dataclasses import astuple


x,y = v2
x,y # (10, 20)
#a,b = v3
#a,b #error

#use astuple()
a,b = astuple(v3)
a,b

(10, 20)

In [ ]:
#or 
@dataclass
class Vector:
    x: int
    y: int

    def __iter__(self):
        # Yield the fields in the order you want them unpacked
        yield self.x
        yield self.y

v4 = Vector(10, 20)

x, y = v4
x,y

(10, 20)

In [29]:
v5 = Vector(10, 20)

In [49]:
class Vector:
    def __init__(self, lat, lon):
        self.lat = lat
        self.lon = lon

vv1 = Vector(10, 20)
vv2 = Vector(10, 20)


In [50]:
vv1 == vv2

False

In [51]:
vv1.lat == vv2.lat

True

@dataclass fices the problem

In [31]:
v4 == v5

True

In Python, the default behavior of a class is to be "selfish." When you create a standard class , Python doesn't automatically look at the data inside to decide if two objects are equal. Instead, it checks their Identity (where they live in memory).

The Default Rule: Unless you tell it otherwise, Python assumes that two different instances of a class are unique individuals, like two identical twins. They look the same, but they are not the same person.

The Missing Method: To make == work, a class needs a special method called __eq__.

In a standard class: __eq__ is inherited from the base object class, which just compares memory addresses. Since objects were created separately, they have different addresses.

In a @dataclass: The decorator automatically writes the __eq__ method for you. It tells Python: "Don't look at the memory address; look at the values of lat and lon."



The term "Code Smell" was coined by Kent Beck and popularized by Martin Fowler. It doesn't mean the code is "broken" (it will run fine); it means the design is weak and will likely cause bugs or headaches in the future.

Why is a Data-Only Class "Smelly"?
When a class has data but no behavior, it violates the Principle of Encapsulation. Encapsulation says: "The data and the rules for that data should live in the same box."

In [33]:
@dataclass
class Employee:
    name: str
    hourly_rate: float
    hours_worked: float

def pay(emp:Employee):
    return f'{emp.name} salary: {emp.hourly_rate} * {emp.hours_worked}'

Logic Duplication: If you need to calculate pay in three different parts of your app (the UI, the PDF exporter, and the Bank API), you will end up writing rate * hours three times.

Maintenance Nightmare: If the rule changes (e.g., "Overtime is 1.5x after 40 hours"), you have to find every single place in your code where that math happens and update it. If you miss one, your app has a bug.

In [34]:
@dataclass
class Employee:
    name: str
    hourly_rate: float
    hours_worked: float

    def calculate_total_pay(self) -> float:
        # The logic is now "Encapsulated"
        if self.hours_worked > 40:
            overtime = self.hours_worked - 40
            return (40 * self.hourly_rate) + (overtime * self.hourly_rate * 1.5)
        return self.hourly_rate * self.hours_worked

# Now, anyone can just call: emp.calculate_total_pay()
# Now, the Employee class is not passive

TypedDict. This is the "ultimate" Duck Typing tool. A TypedDict isn't a real class; it's just a regular Python Dictionary with some "rules" for the code editor.

In [ ]:
# This is NOT a class. It's a dictionary with a "label".
from typing import TypedDict
class CoordinateDict(TypedDict):
    lat: float
    lon: float

moscow = {'lat': 55.76, 'lon': 37.62}
location = {'lat': 55.76, 'lon': 37.62}

print(moscow == location) # True!

# At RUNTIME, Python sees this:
# moscow = {'lat': 55.76, 'lon': 37.62}
# location = {'lat': 55.76, 'lon': 37.62}

True


Data Classes / NamedTuples: These exist as "Concrete Classes" at runtime. You can check type(obj) and see exactly what they are.

TypedDict: This disappears at runtime.

TypedDict is purely for "Static Analysis" (the red squiggly lines in your code editor). Once you click "Run," a TypedDict is just a plain old Python Dictionary.

The Difference:

Duck Typing (Runtime): "I'll try to call .lat and see if it crashes."

TypedDict (Pre-Runtime): "My editor warned me that this dictionary is missing a lat key before I even ran the code."


If you use TypedDict, you are basically using a "naked" dictionary. This is often the ultimate "Data Class" code smell

The "Bottom Line": Use @dataclass when you want a real object that can hold its own logic. Use TypedDict only when you are forced to work with raw dictionaries (like JSON from a web API) but want your code editor to help you avoid typos

@dataclass (c3): This is a full-featured object. It has a hidden __dict__ where it stores your data. This makes it slightly heavier but much more flexible for adding methods (solving the "smell")

## Type Hints 101

The first thing you need to know about type hints is that they are not enforced at all
by the Python bytecode compiler and interpreter.
Think about Python type hints as “documentation that can be verified by IDEs and
type checkers.”
That’s because type hints have no impact on the runtime behavior of Python pro‐
grams

What is Pydantic? (The Short Version)
Think of Pydantic as a strict bouncer for your data. When your app receives data (like from a user filling out a form or an API), Pydantic looks at your type hints and says: "Wait, you said age should be an int, but you sent the word 'twenty'. I'm throwing an error." It automatically validates data so your app doesn't crash later.

In [ ]:
class Demo:
    a: int
    b: float=1.1
    c = 'spam'
Demo.__annotations__

{'a': int, 'b': float}

Analysis of the Demo
Variable a: Because it is defined with a type hint (a: int) but no value, it becomes an entry in the __annotations__ dictionary. However, no actual attribute named a is created in the class itself at runtime.

Variable b: Since it has both a type hint and a value (b: float = 1.1), it is saved as an annotation and also exists as a standard class attribute with the value 1.1.

Variable c: Because it lacks a type hint (c = 'spam'), it is treated as a "plain old class attribute" and is not included in the __annotations__ dictionary at all.


In [7]:
class QudratQuestion:
    # Pattern 1: Annotation Only
    question_text: str
    
    # Pattern 2: Annotation + Default
    points: int = 10
    
    # Pattern 3: Plain Attribute
    exam_type = "Qudrat GAT"

In [8]:
# 1. Create an empty instance
q1 = QudratQuestion()

# 2. Manually assign the text
q1.question_text = "If 3x = 12, what is x?"

print(q1.question_text) # Output: If 3x = 12, what is x?
print(q1.points)        # Output: 10 (from the default)

If 3x = 12, what is x?
10


In [ ]:
from dataclasses import dataclass

@dataclass
class QudratQuestion:
    question_text: str       # Required argument!   # Instance Attributes
    points: int = 10         # Optional argument (defaults to 10) #  Instance Attributes
    exam_type = "Qudrat GAT" # Ignored by dataclass (no type hint)  # Class Attributes

# Now you CAN pass arguments exactly like you wanted to:
q2 = QudratQuestion(question_text="What is the synonym of 'Abundant'?")

print(q2.question_text)
# Output: What is the synonym of 'Abundant'?

print(q2.points)
# Output: 10

What is the synonym of 'Abundant'?
10


In [10]:
from typing import NamedTuple

class NahwQuestion(NamedTuple):
    # Pattern 1: Annotation Only -> Becomes a REQUIRED argument
    sentence: str
    
    # Pattern 2: Annotation + Default -> Becomes an OPTIONAL argument
    points: int = 5
    
    # Pattern 3: Plain Attribute -> Ignored as an argument, stays a background constant
    subject = "Arabic Grammar"

# You can pass arguments perfectly, just like a dataclass!
q3 = NahwQuestion(sentence="ذهب الطالب إلى المدرسة")

print(q3.sentence)
# Output: ذهب الطالب إلى المدرسة

print(q3.points)
# Output: 5

ذهب الطالب إلى المدرسة
5


If the eq and frozen arguments are both True, @dataclass produces a suitable
__hash__ method, so the instances will be hashable. The generated __hash__ will use
data from all fields that are not individually excluded using a field option we’ll see in
“Field Options” on page 180. If frozen=False (the default), @dataclass will set
__hash__ to None, signalling that the instances are unhashable, therefore overriding
__hash__ from any superclass.

In [11]:
@dataclass
class Q:
    question: str
    choices: list=[]

question1 = Q('how are u',[1,2,3,4])

ValueError: mutable default <class 'list'> for field choices is not allowed: use default_factory

In [13]:
from dataclasses import field
@dataclass
class Q:
    question: str
    choices: list=field(default_factory=list)

question1 = Q('how are u',[1,2,3,4])
question1

Q(question='how are u', choices=[1, 2, 3, 4])

The default_factory parameter lets you provide a function, class, or any other call‐
able, which will be invoked with zero arguments to build a default value each time an
instance of the data class is created. This way, each instance of ClubMember will have
its own list—instead of all instances sharing the same list from the class, which is
rarely what we want and is often a bug.

In [15]:
from dataclasses import field
@dataclass
class Q:
    question: str
    choices: list[str]=field(default_factory=list)

question1 = Q('how are u',[1,2,3,4])
question1

Q(question='how are u', choices=[1, 2, 3, 4])

The "Type Hint" is Metadata, Not a Rule
In your code, list[str] is saved as an annotation. Python's philosophy is "Dynamic Typing," meaning it prioritizes flexibility over strictness. It assumes you, the developer, know what you are doing. If you pass a list of integers to a slot marked for strings, Python won't stop you unless you use a library specifically designed to do so.

In [18]:
from pydantic import BaseModel

class Q(BaseModel):
    question: str
    choices: list[str]

# This will now throw a ValidationError because 1, 2, 3 are not strings!
question1 = Q(question='how are u', choices=[1, 2, 3, 4])

ValidationError: 4 validation errors for Q
choices.0
  Input should be a valid string [type=string_type, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
choices.1
  Input should be a valid string [type=string_type, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
choices.2
  Input should be a valid string [type=string_type, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
choices.3
  Input should be a valid string [type=string_type, input_value=4, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

Core Configuration Options

default: This sets a standard default value for the field (like False or 0). It takes the place of the usual = value in the annotation because the field() call is already occupying that spot.

default_factory: This is a function (like list or dict) used to generate a fresh default value every time a new instance is created. You used this in your choices: list[str] example to avoid the "mutable default argument" bug.

init: If set to False, this attribute will not be an argument in the __init__ method. You would use this for internal variables that the user shouldn't set manually.

repr: If set to False, this field will be hidden when you print() the object. This is useful for sensitive data like passwords or very long lists.

Advanced Logic Options
compare: Determines if this field is used when comparing two objects (e.g., obj1 == obj2).

hash: Controls whether this field is used to calculate a unique hash for the object.

metadata: A place to store your own custom information about the field. The @dataclass itself ignores this, but other tools (like your GAT platform's grading logic) could read it to know how to process that specific field.

When you use @dataclass, Python automatically generates the __init__ method for you.

Sometimes you want extra logic after initialization (validation, computed fields, etc.).

In [23]:
@dataclass
class ClubMember:
    name: str

@dataclass
class Hacker(ClubMember):
    handles= set()  ## FOCUS we did this with no type anotation of default factory because we want a set for the whole class (no multiplication)
    handle : str=""

    def __post_init__(self):
        cls = self.__class__ #Just stores the class in a variable
        if self.handle == "":
            self.handle = self.name.split()[0]
        if self.handle in cls.handles:
            msg = f'{self.handle!r} already exist'
            raise ValueError(msg)
        cls.handles.add(self.handle)

Hacker(name='mike momo')
Hacker.handles

{'mike'}

When using @dataclass, any attribute that has a type annotation is automatically treated as an instance field (each object gets its own copy). In the example, all_handles = set() was intended to be a class-level attribute shared by all instances, but when running Mypy, it complains because the variable has no type annotation. If we add a normal annotation like all_handles: set[str] = set(), @dataclass will mistakenly treat it as an instance attribute, which is not what we want. The solution  is to mark it explicitly as a class variable using ClassVar from typing, like all_handles: ClassVar[set[str]] = set(). This tells both the type checker and @dataclass that the variable belongs to the class itself (shared by all objects) and should not be turned into a dataclass field.

In [ ]:
from typing import ClassVar

handles: ClassVar[set[str]] = set()

# handles: set[str] = set()      → instance attribute
# handles: ClassVar[set[str]] = set()  → class attribute (shared)

Sometimes you may need to pass arguments to __init__ that are not instance fields.
Such arguments are called init-only variables by the dataclasses documentation


## Initialization Variables That Are Not Fields
An InitVar in a dataclass is a parameter that is accepted by __init__ but is NOT stored as an instance attribute. It exists only during initialization so you can use it inside __post_init__ to help compute real fields.

Why this is useful

Sometimes you need extra information to build the object, but that information should not become part of the object’s state.

Example:
You might need a database connection just to fetch a value during initialization.

You don't want the object to permanently store the database object.

In [ ]:
from dataclasses import InitVar


@dataclass
class Data:
    i: int
    j: int= None
    database: InitVar[DatabaseType] = None

    def __post_init__(self, database):
        if self.j is None and database is not None:
            self.j = database.lookup('j')

In [ ]:
from dataclasses import dataclass, field, fields
from typing import Optional
from enum import Enum, auto
from datetime import date

# 1. Defining Type-Safe Categories
class ResourceType(Enum):
    BOOK = auto()  #Inside an Enum, every name (like BOOK) needs a value behind it (like 1).
    EBOOK = auto()
    VIDEO = auto()

# 2. Building the Data Class
@dataclass
class Resource:
    """Media resource description."""
    identifier: str
    title: str = '<untitled>'
    
    # We must use default_factory for mutable types like lists
    creators: list[str] = field(default_factory=list)
    date: Optional[date] = None
    type: ResourceType = ResourceType.BOOK
    description: str = ''
    language: str = ''
    subjects: list[str] = field(default_factory=list)

    # 3. Customizing the Output Representation
    def __repr__(self):
        cls = self.__class__
        cls_name = cls.__name__
        indent = ' ' * 4
        res = [f'{cls_name}(']  # Starts the list with "Resource("
        
        for f in fields(cls):
            value = getattr(self, f.name)
            res.append(f'{indent}{f.name} = {value!r},')
            
        res.append(')')
        return '\n'.join(res)

# --- How it is used  ---
book = Resource(
    '978-0-13-475759-9', 
    'Refactoring, 2nd Edition',
    ['Martin Fowler', 'Kent Beck'], 
    date(2018, 11, 19),
    ResourceType.EBOOK, 
    'Improving the design of existing code', 
    'EN',
    ['computer programming', 'OOP']
)

print(book)

Resource(
    identifier = '978-0-13-475759-9',
    title = 'Refactoring, 2nd Edition',
    creators = ['Martin Fowler', 'Kent Beck'],
    date = datetime.date(2018, 11, 19),
    type = <ResourceType.EBOOK: 2>,
    description = 'Improving the design of existing code',
    language = 'EN',
    subjects = ['computer programming', 'OOP'],
)


Enum stands for Enumeration. It is used when you have a variable that should only ever be one of a few specific things.

In [33]:
movie = Resource(identifier="456", type="vidio")
movie

Resource(
    identifier = '456',
    title = '<untitled>',
    creators = [],
    date = None,
    type = 'vidio',
    description = '',
    language = '',
    subjects = [],
)

Python's "Blind Trust" (The lack of Runtime Enforcement)
In Python, Type Hints (like type: ResourceType) are just suggestions. They are like a sign on a door that says "Employees Only," but the door isn't actually locked.

When you ran:
movie = Resource(identifier="456", type="vidio")

In [35]:
if movie.type == ResourceType.VIDEO:
    print("Playing video...")
else:
    print("Resource not recognized.")

Resource not recognized.


In [34]:
movie2 = Resource(identifier="456", type=ResourceType.BOOKKK)
movie2

AttributeError: type object 'ResourceType' has no attribute 'BOOKKK'

In [38]:
search_term = input("What do you want to see? ") # User types 'title'

# You can't do: movie.search_term 
# (Python will look for a variable actually named 'search_term')

# You MUST do:
value = getattr(book, search_term)
print(value) # Output: 'Refactoring'

Refactoring, 2nd Edition


getattr is just a way to do object.attribute, but using a string instead of typing the attribute name directly. We use it in the __repr__ because we want a single piece of code that can look up any field name that exists in the class.

## Pattern Matching Class Instances - Structural Pattern Matching

### Simple Class Patterns (The "Constructor" Look)

The syntax looks exactly like you are creating an object, but you are actually deconstructing it.

In [40]:
x= 1.1
match x:
    case float():
        print('decimal number')



decimal number


In [41]:
match x:
    case float:  # NO PARENTHESES
        print("This will ALWAYS run")

This will ALWAYS run


### Keyword Patterns Work

In [43]:
import typing

class City(typing.NamedTuple):
    continent: str
    name: str
    country: str

cities = [
    City('Asia', 'Tokyo', 'JP'),
    City('Asia', 'Delhi', 'IN'),
    City('North America', 'New York', 'US'),
    City('South America', 'São Paulo', 'BR'),
]

def analyze_cities(city_list):
    for city in city_list:
        # This is where Keyword Pattern Matching happens
        match city:
            # Pattern 1: Match by specific value AND capture data
            case City(continent='Asia', name=city_name):
                print(f" Asian City Found: {city_name}")

            # Pattern 2: Match by a different continent, ignore the name
            case City(continent='North America'):
                print(f" North American City detected: {city.name}")

            # Pattern 3: Catch-all for anything else
            case _:
                print(" Other region...")

analyze_cities(cities)

 Asian City Found: Tokyo
 Asian City Found: Delhi
 North American City detected: New York
 Other region...


### Positional Class Patterns

the "shorthand" version of what we just discussed. Instead of typing continent='Asia', you just type 'Asia'. But for this to work, Python needs to know which "slot" belongs to which variable.

That is where the "Magic" of __match_args__ comes in.

In [44]:
City.__match_args__


('continent', 'name', 'country')

In [47]:
def match_asian_countries_pos():
    results = []
    for city in cities:
        match city:
            case City('Asia', _, country=cc):  
                results.append(cc)
    return results
match_asian_countries_pos()

['JP', 'IN']

because @dataclass and NamedTuple make it so easy to create classes without methods. Because they automatically provide __match_args__, they "invite" you to write external match statements instead of internal methods.

How ever, "Tell, Don't Ask."

The "Smelly" Way (Asking): You reach into the City object, grab its data, and you (the external function) decide if it's Asian.

The "Optimal" Way (Telling): You tell the City object: "Hey, give me your tax rate," and the city handles its own internal logic.

In [48]:
class City(NamedTuple):
    continent: str
    name: str

    @property
    def is_asian(self):
        # This logic is now ENCAPSULATED inside the class
        return self.continent == 'Asia'

# --- Using it ---
tokyo = City('Asia', 'Tokyo')

# Without @property, you'd type: tokyo.is_asian()
# WITH @property, you type:
if tokyo.is_asian: 
    print("Welcome to Asia!")

Welcome to Asia!


The @property decorator is a "magic" tool that makes a method look and act like a simple variable (an attribute) to the outside world.

In [ ]:
from enum import Enum
from pydantic import BaseModel, ValidationError

# 1. The Enum defines the ALLOWED choices
class Status(Enum):
    ACTIVE = "active"
    INACTIVE = "inactive"

# 2. Pydantic defines the STRUCTURE and ENFORCES the Enum
class User(BaseModel):
    name: str
    status: Status  # This MUST be a valid Status enum

# --- This works ---
u1 = User(name="Gemini", status=Status.ACTIVE)

# --- This CRASHES (The Guard works!) ---
try:
    u2 = User(name="Robot", status="waiting") # Not in the Enum!
except ValidationError as e:
    print("Caught the error!")